In [ ]:
!pip install diffusers

import torch
import numpy as np
import matplotlib.pyplot as plt
import PIL.Image
import tqdm
from diffusers import DDPMPipeline, UNet2DModel, DDPMScheduler, DDIMScheduler

image_pipe = DDPMPipeline.from_pretrained("google/ddpm-celebahq-256")
image_pipe.to("cuda")
images = image_pipe().images
images[0]

image_pipe

repo_id = "google/ddpm-church-256"
model = UNet2DModel.from_pretrained(repo_id)
model
model.config

model_random = UNet2DModel(**model.config)
model_random.save_pretrained("my_model")

!ls my_model

model_random = UNet2DModel.from_pretrained("my_model")

torch.manual_seed(0)
noisy_sample = torch.randn(
    1, model.config.in_channels, model.config.sample_size, model.config.sample_size
)
noisy_sample.shape

with torch.no_grad():
    noisy_residual = model(sample=noisy_sample, timestep=2).sample
noisy_residual.shape

scheduler = DDPMScheduler.from_config(repo_id)
scheduler.config
scheduler.save_config("my_scheduler")
new_scheduler = DDPMScheduler.from_config("my_scheduler")

less_noisy_sample = scheduler.step(
    model_output=noisy_residual, timestep=2, sample=noisy_sample
).prev_sample
less_noisy_sample.shape

def display_sample(sample, i):
    image_processed = sample.cpu().permute(0, 2, 3, 1)
    image_processed = (image_processed + 1.0) * 127.5
    image_processed = image_processed.numpy().astype(np.uint8)
    image_pil = PIL.Image.fromarray(image_processed[0])
    display(f"Image at step {i}")
    display(image_pil)

model.to("cuda")
noisy_sample = noisy_sample.to("cuda")

sample = noisy_sample
for i, t in enumerate(tqdm.tqdm(scheduler.timesteps)):
    with torch.no_grad():
        residual = model(sample, t).sample
    sample = scheduler.step(residual, t, sample).prev_sample
    if (i + 1) % 50 == 0:
        display_sample(sample, i + 1)

scheduler = DDIMScheduler.from_config(repo_id)
scheduler.set_timesteps(num_inference_steps=50)

sample = noisy_sample
for i, t in enumerate(tqdm.tqdm(scheduler.timesteps)):
    with torch.no_grad():
        residual = model(sample, t).sample
    sample = scheduler.step(residual, t, sample).prev_sample
    if (i + 1) % 10 == 0:
        display_sample(sample, i + 1)

scheduler_cifar10 = DDPMScheduler.from_pretrained("google/ddpm-cifar10-256")
scheduler_butterfly = DDPMScheduler.from_pretrained("jonathanho/ddpm-butterflies-32px")

betas_cifar10 = scheduler_cifar10.betas.cpu()
alphas_cumprod_cifar10 = scheduler_cifar10.alphas_cumprod.cpu()

betas_butterfly = scheduler_butterfly.betas.cpu()
alphas_cumprod_butterfly = scheduler_butterfly.alphas_cumprod.cpu()

plt.figure(figsize=(10, 4))
plt.plot(betas_cifar10, label="CIFAR-10 (google/ddpm-cifar10-256)")
plt.plot(betas_butterfly, label="Butterfly (jonathanho/ddpm-butterflies-32px)")
plt.title("Beta Schedule")
plt.xlabel("Timestep")
plt.ylabel("Beta")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(alphas_cumprod_cifar10, label="CIFAR-10 (google/ddpm-cifar10-256)")
plt.plot(alphas_cumprod_butterfly, label="Butterfly (jonathanho/ddpm-butterflies-32px)")
plt.title("Alpha Cumprod")
plt.xlabel("Timestep")
plt.ylabel("Alphas Cumprod")
plt.legend()
plt.grid(True)
plt.show()


Output hidden; open in https://colab.research.google.com to view.

Beta Schedule 類型與比較

google/ddpm-cifar10-256 使用的是 linear beta schedule，也就是 beta 值在 timestep 上線性增加。
jonathanho/ddpm-butterflies-32px 使用的是 cosine beta schedule，這種設計在前期保留更多原始資訊，在後期快速加強雜訊。

差異比較：

beta 的變化：
   Linear：穩定上升（近似直線），如 CIFAR-10。
   Cosine：初期上升緩慢、後期急速上升（曲線彎曲），如 Butterfly 模型。

alphas_cumprod 曲線：
   Linear 下降快，模型在後期訊息幾乎全雜訊。
   Cosine 下降慢，在多數 timestep 保留更多圖像結構，有助於生成高品質圖。

總結:
cosine schedule 常被認為在圖像品質與收斂穩定性上優於線性排程，因此 jonathanho 提出的 cosine schedule 成為後續改進的基礎之一。
